# 01 · The Odds API — full v4 endpoint run

**Purpose.** Exercise EVERY relevant Odds API v4 read endpoint under a
hard credit budget, land the payloads in the raw layer, and build the
bronze/silver odds tables the matching + convergence notebooks read.

**Contents**: [registry](#registry) · [guard](#guard) ·
[sport-key discovery](#sportkey) · [endpoint runs](#runs) ·
[bronze/silver builds](#builds) · [call log & quota](#log) ·
[findings](#findings)

**What to change**: `MAX_CREDITS` (budget), `DRY_RUN` (estimate only),
`RUN_HISTORICAL` (expensive 10× endpoints), `p.offline` (cache only).

In [1]:
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
JB = next(q for q in [_here, *_here.parents] if (q / "lib" / "bootstrap.py").exists())
sys.path.insert(0, str(JB))

import pandas as pd
import polars as pl
import lib.bootstrap as bt
import lib.config as cfg
import lib.oddsapi as oa
import lib.storage as st
import lib.ids as ids
import lib.fairvalue as fv

manifest = bt.run_manifest("01_odds_api_full_run")

/Users/andrewdoherty/Desktop/Coding/World Cup Alpha/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Parameters — the knobs for THIS notebook

In [2]:
p = cfg.load_params()
MAX_CREDITS = p.max_credits      # hard per-run budget (default 25)
DRY_RUN = False                  # True → estimate costs, no live calls
RUN_HISTORICAL = False           # 10×-cost endpoints; flip deliberately
CACHE_MAX_AGE_S = 1800           # reuse raw snapshots younger than 30 min
REGIONS, MARKETS = "uk", "h2h,totals"
print(f"budget {MAX_CREDITS} credits · dry_run={DRY_RUN} · offline={p.offline}")

budget 25 credits · dry_run=False · offline=False


<a id="registry"></a>
## 1 · Endpoint registry
Complete v4 read surface with the DOCUMENTED credit-cost formula per
endpoint (estimates — the guard trues up from response headers).

In [3]:
registry = pd.DataFrame([
    {"endpoint": e.key, "path": e.path, "cost (documented)": e.cost_note,
     "what it serves": e.doc}
    for e in oa.ENDPOINTS.values()])
registry

,endpoint,path,cost (documented),what it serves
0,sports,/sports,free,All in-season sports (all=true adds out-of-sea...
1,events,/sports/{sport_key}/events,free,"Upcoming/live events for a sport, no odds."
2,odds,/sports/{sport_key}/odds,regions × markets (featured markets only),Bulk featured odds for every upcoming event.
3,event_markets,/sports/{sport_key}/events/{event_id}/markets,regions (documented as counted like one market...,Which markets each bookmaker currently offers ...
4,event_odds,/sports/{sport_key}/events/{event_id}/odds,regions × unique markets — the only route to p...,"Odds for ONE event; supports btts, alternates,..."
5,scores,/sports/{sport_key}/scores,1 live-only; 2 with daysFrom (adds recently co...,Live + recently completed scores.
6,participants,/sports/{sport_key}/participants,1,Team list for a sport.
7,historical_events,/historical/sports/{sport_key}/events,"1 (metadata snapshot, no odds)",Event list as of a past `date` (ISO).
8,historical_odds,/historical/sports/{sport_key}/odds,10 × regions × markets — EXPENSIVE,Bulk featured odds snapshot as of a past `date`.
9,historical_event_odds,/historical/sports/{sport_key}/events/{event_i...,10 × regions × unique markets — EXPENSIVE,One event's odds (incl. props) as of a past `d...


<a id="guard"></a>
## 2 · Quota guard
Central budget: every call estimates first, spends only if affordable,
logs everything (live / cache / dry-run / skip + reason).

In [4]:
guard = oa.QuotaGuard(max_credits=MAX_CREDITS, dry_run=DRY_RUN)

<a id="sportkey"></a>
## 3 · World Cup sport key — discovered, validated, never hardcoded

In [5]:
sport_key = None
try:
    disc = oa.discover_wc_sport_key(guard, offline=p.offline)
    sport_key = disc["sport_key"]
    print("validated sport key:", sport_key)
    pd.DataFrame(disc["candidates"])
except oa.SkippedCall as e:
    print("sport-key discovery skipped:", e)

validated sport key: soccer_fifa_world_cup


<a id="runs"></a>
## 4 · Run every endpoint (skips recorded, never silent)
Each block is one endpoint; failures/skips append to the call log with the
reason. Historical endpoints run in DRY-RUN unless `RUN_HISTORICAL=True`
so their cost is visible without spending 10× credits.

In [6]:
payloads = {}

def attempt(key, **params):
    try:
        payload, snap, meta = oa.fetch(key, guard, offline=p.offline,
                                       cache_max_age_s=CACHE_MAX_AGE_S, **params)
        payloads[key] = payload
        n = len(payload) if isinstance(payload, list) else 1
        print(f"{key:24s} OK  ({n} objects) → {snap}")
    except oa.SkippedCall as e:
        print(f"{key:24s} SKIP: {e}")

if sport_key:
    attempt("events", sport_key=sport_key)                       # free
    attempt("odds", sport_key=sport_key, regions=REGIONS,        # 2 credits
            markets=MARKETS, oddsFormat="decimal")
    attempt("scores", sport_key=sport_key, daysFrom=2)           # 2 credits
    attempt("participants", sport_key=sport_key)                 # 1 credit

events                   OK  (8 objects) → theoddsapi/sports_soccer_fifa_world_cup_events/20260704T131807Z__41fc6e4466
odds                     OK  (8 objects) → theoddsapi/sports_soccer_fifa_world_cup_odds/20260704T131808Z__e58a3f96ef
scores                   OK  (14 objects) → theoddsapi/sports_soccer_fifa_world_cup_scores/20260704T131809Z__15f14b3a03
participants             OK  (91 objects) → theoddsapi/sports_soccer_fifa_world_cup_participants/20260704T131810Z__41fc6e4466


In [7]:
# Per-event endpoints: nearest upcoming event from /events (if any)
event_id_src = None
if payloads.get("events"):
    evs = sorted(payloads["events"], key=lambda e: e.get("commence_time") or "")
    upcoming = [e for e in evs if (e.get("commence_time") or "") >= manifest["run_utc"]]
    if upcoming:
        event_id_src = upcoming[0]["id"]
        print("nearest upcoming event:", upcoming[0].get("home_team"), "vs",
              upcoming[0].get("away_team"), upcoming[0].get("commence_time"))
if sport_key and event_id_src:
    attempt("event_markets", sport_key=sport_key, event_id=event_id_src,
            regions=REGIONS)                                     # 1 credit
    attempt("event_odds", sport_key=sport_key, event_id=event_id_src,
            regions=REGIONS, markets="btts,draw_no_bet",         # 2 credits
            oddsFormat="decimal")
    # player props are only served per-event; many books don't post them for
    # every WC match — a 422/empty response is recorded, not hidden
    attempt("event_odds", sport_key=sport_key, event_id=event_id_src,
            regions=REGIONS, markets="player_goal_scorer_anytime",
            oddsFormat="decimal")
else:
    guard.log(endpoint="event_markets", mode="skip",
              est_credits=1, reason="no upcoming event id available this run")
    guard.log(endpoint="event_odds", mode="skip",
              est_credits=2, reason="no upcoming event id available this run")
    print("per-event endpoints skipped: no upcoming event id")

nearest upcoming event: Canada vs Morocco 2026-07-04T17:00:00Z
event_markets            OK  (1 objects) → theoddsapi/sports_soccer_fifa_world_cup_events_9c7073ae2c29ee4881bb695f92168c68_markets/20260704T131811Z__96ee147c6b
event_odds               OK  (1 objects) → theoddsapi/sports_soccer_fifa_world_cup_events_9c7073ae2c29ee4881bb695f92168c68_odds/20260704T131811Z__7a24536442
event_odds               OK  (1 objects) → theoddsapi/sports_soccer_fifa_world_cup_events_9c7073ae2c29ee4881bb695f92168c68_odds/20260704T131812Z__5b8e5b8700


In [8]:
# Historical endpoints (10× multiplier) — coded and runnable; DRY-RUN by
# default so the cost is demonstrated without spending.
hist_guard = guard if RUN_HISTORICAL else oa.QuotaGuard(
    max_credits=MAX_CREDITS, dry_run=True)
hist_date = "2026-06-20T12:00:00Z"   # inside our sportsbook-snapshot window
if sport_key:
    for key, params in [
            ("historical_events", {"date": hist_date}),
            ("historical_odds", {"date": hist_date, "regions": REGIONS,
                                 "markets": "h2h"}),
            ("historical_event_odds", {"date": hist_date, "regions": REGIONS,
                                       "markets": "h2h",
                                       "event_id": event_id_src or "unknown"})]:
        try:
            payload, snap, _ = oa.fetch(key, hist_guard, offline=p.offline,
                                        sport_key=sport_key, **params)
            payloads[key] = payload
            print(f"{key:24s} OK → {snap}")
        except oa.SkippedCall as e:
            print(f"{key:24s} SKIP: {e}")
    if not RUN_HISTORICAL:
        for c in hist_guard.calls:
            guard.calls.append({**c, "reason": (c.get("reason") or "")
                                + " [RUN_HISTORICAL=False]"})

historical_events        SKIP: historical_events: dry-run (est 1 credits)
historical_odds          SKIP: historical_odds: dry-run (est 10 credits)
historical_event_odds    SKIP: historical_event_odds: dry-run (est 10 credits)


<a id="builds"></a>
## 5 · Bronze + silver builds
Bronze uses PRODUCTION parsing (`wca.data.theoddsapi._parse_events`) so
rows here are shaped exactly like production's. Silver adds canonical IDs
and per-book de-vigged probabilities.

In [9]:
bronze_odds = pl.DataFrame()
if payloads.get("odds"):
    pdf = oa.parse_odds_payload(payloads["odds"])
    bronze_odds = pl.from_pandas(pdf)
    st.save_dataset(bronze_odds, "bronze", "oddsapi_odds",
                    inputs=[c.get("snapshot", "") for c in guard.calls
                            if c.get("endpoint") == "odds"],
                    notebook="01", note=f"regions={REGIONS} markets={MARKETS}")
    prof = st.profile_frame(bronze_odds, "bronze oddsapi_odds")
    print(f"bronze oddsapi_odds: {prof.attrs['rows']} rows")
elif st.dataset_path("bronze", "oddsapi_odds").exists():
    bronze_odds = st.load_dataset("bronze", "oddsapi_odds")
    print(f"live pull unavailable — reusing prior bronze ({bronze_odds.height} rows)")
else:
    print("no odds data this run and no prior bronze — downstream notebooks "
          "will use their PM/model fallbacks")
st.profile_frame(bronze_odds, "bronze") if bronze_odds.height else None

bronze oddsapi_odds: 602 rows


,column,dtype,null_rate
0,event_id,String,0.0000
1,commence_time,"Datetime(time_unit='ns', time_zone='UTC')",0.0000
2,home_team,String,0.0000
3,away_team,String,0.0000
4,bookmaker_key,String,0.0000
5,bookmaker_title,String,0.0000
6,retrieved_at,"Datetime(time_unit='ns', time_zone='UTC')",0.0000
7,market,String,0.0000
8,outcome_name,String,0.0000
9,outcome_description,String,1.0000


In [10]:
silver_quotes = pl.DataFrame()
if bronze_odds.height:
    import datetime as dt
    rows = []
    for key, grp in bronze_odds.group_by(
            ["event_id", "bookmaker_key", "market"], maintain_order=True):
        g = grp.to_dicts()
        odds = [r["decimal_odds"] for r in g]
        if not odds or any(o is None or o <= 1.0 for o in odds):
            continue
        try:
            devigged = fv.devig(odds, p.devig_method)
        except Exception:
            continue
        ko = g[0]["commence_time"]
        mkt = g[0]["market"]
        ev = ids.event_id(g[0]["home_team"], g[0]["away_team"], ko)
        mk = ids.market_id(ev, "1x2" if mkt == "h2h" else mkt,
                           line=g[0].get("outcome_point"),
                           settlement=ids.S_90MIN)
        for r, q in zip(g, devigged):
            rows.append({
                "event_id": ev, "market_id": mk,
                "outcome_id": ids.outcome_id(mk, r["outcome_name"]),
                "source": "theoddsapi", "source_event_id": r["event_id"],
                "bookmaker": r["bookmaker_key"], "market_src": mkt,
                "outcome": r["outcome_name"],
                "decimal_odds": r["decimal_odds"], "line": r.get("outcome_point"),
                "implied_prob": 1 / r["decimal_odds"], "devig_prob": float(q),
                "devig_method": p.devig_method,
                "kickoff_utc": ko, "retrieved_utc": r["retrieved_at"],
            })
    silver_quotes = pl.DataFrame(rows)
    st.save_dataset(silver_quotes, "silver", "sportsbook_quotes",
                    inputs=["bronze/oddsapi_odds"], notebook="01")
    print(f"silver sportsbook_quotes: {silver_quotes.height} rows "
          f"({silver_quotes['event_id'].n_unique()} events, "
          f"{silver_quotes['bookmaker'].n_unique()} books)")
    display(silver_quotes.head(6).to_pandas())
else:
    print("silver sportsbook_quotes not (re)built this run")

silver sportsbook_quotes: 602 rows (8 events, 19 books)


,event_id,market_id,outcome_id,source,source_event_id,bookmaker,market_src,outcome,decimal_odds,line,implied_prob,devig_prob,devig_method,kickoff_utc,retrieved_utc
0,wc2026:canada__morocco__2026-07-04T17Z,wc2026:canada__morocco__2026-07-04T17Z|1x2||FT...,wc2026:canada__morocco__2026-07-04T17Z|1x2||FT...,theoddsapi,9c7073ae2c29ee4881bb695f92168c68,grosvenor,h2h,Canada,5.00,NaN,0.200000,0.178786,shin,2026-07-04 17:00:00+00:00,2026-07-04 13:17:51+00:00
1,wc2026:canada__morocco__2026-07-04T17Z,wc2026:canada__morocco__2026-07-04T17Z|1x2||FT...,wc2026:canada__morocco__2026-07-04T17Z|1x2||FT...,theoddsapi,9c7073ae2c29ee4881bb695f92168c68,grosvenor,h2h,Morocco,1.73,NaN,0.578035,0.550119,shin,2026-07-04 17:00:00+00:00,2026-07-04 13:17:51+00:00
2,wc2026:canada__morocco__2026-07-04T17Z,wc2026:canada__morocco__2026-07-04T17Z|1x2||FT...,wc2026:canada__morocco__2026-07-04T17Z|1x2||FT...,theoddsapi,9c7073ae2c29ee4881bb695f92168c68,grosvenor,h2h,Draw,3.40,NaN,0.294118,0.271094,shin,2026-07-04 17:00:00+00:00,2026-07-04 13:17:51+00:00
3,wc2026:canada__morocco__2026-07-04T17Z,wc2026:canada__morocco__2026-07-04T17Z|totals|...,wc2026:canada__morocco__2026-07-04T17Z|totals|...,theoddsapi,9c7073ae2c29ee4881bb695f92168c68,grosvenor,totals,Over,2.16,2.5,0.462963,0.430277,shin,2026-07-04 17:00:00+00:00,2026-07-04 13:17:51+00:00
4,wc2026:canada__morocco__2026-07-04T17Z,wc2026:canada__morocco__2026-07-04T17Z|totals|...,wc2026:canada__morocco__2026-07-04T17Z|totals|...,theoddsapi,9c7073ae2c29ee4881bb695f92168c68,grosvenor,totals,Under,1.66,2.5,0.602410,0.569723,shin,2026-07-04 17:00:00+00:00,2026-07-04 13:17:51+00:00
5,wc2026:canada__morocco__2026-07-04T17Z,wc2026:canada__morocco__2026-07-04T17Z|1x2||FT...,wc2026:canada__morocco__2026-07-04T17Z|1x2||FT...,theoddsapi,9c7073ae2c29ee4881bb695f92168c68,leovegas,h2h,Canada,5.50,NaN,0.181818,0.170437,shin,2026-07-04 17:00:00+00:00,2026-07-04 13:18:03+00:00


In [11]:
# Events + scores + participants bronze (source-shaped, thin)
for key, name in [("events", "oddsapi_events"), ("scores", "oddsapi_scores"),
                  ("participants", "oddsapi_participants")]:
    if payloads.get(key):
        df = pl.DataFrame([{k: (str(v) if isinstance(v, (dict, list)) else v)
                            for k, v in row.items()} for row in payloads[key]],
                          infer_schema_length=None)
        st.save_dataset(df, "bronze", name, notebook="01",
                        inputs=[c.get("snapshot", "") for c in guard.calls
                                if c.get("endpoint") == key])
        print(f"bronze {name}: {df.height} rows")

bronze oddsapi_events: 8 rows
bronze oddsapi_scores: 14 rows
bronze oddsapi_participants: 91 rows


<a id="log"></a>
## 6 · Full call log + quota accounting
`est_credits` are documented-formula ESTIMATES; `quota_remaining_hdr` is
the provider's own header — the ground truth.

In [12]:
call_log = guard.to_frame()
import lib.plotting as plot
plot.save_table(call_log, "01_oddsapi_call_log")
print(f"estimated credits spent this run: {guard.spent_estimated} / {MAX_CREDITS}")
print(f"provider-reported remaining: {guard.remaining_reported}")
call_log

estimated credits spent this run: 0 / 25
provider-reported remaining: None


,endpoint,mode,est_credits,snapshot,cache_age_s,status,utc,reason
0,sports,cache,0,theoddsapi/sports/20260704T131806Z__ee9e145f73,512.872379,200.0,2026-07-04T13:26:38Z,NaN
1,events,cache,0,theoddsapi/sports_soccer_fifa_world_cup_events...,511.876252,200.0,2026-07-04T13:26:38Z,NaN
2,odds,cache,0,theoddsapi/sports_soccer_fifa_world_cup_odds/2...,510.876655,200.0,2026-07-04T13:26:38Z,NaN
3,scores,cache,0,theoddsapi/sports_soccer_fifa_world_cup_scores...,509.877571,200.0,2026-07-04T13:26:38Z,NaN
4,participants,cache,0,theoddsapi/sports_soccer_fifa_world_cup_partic...,508.877742,200.0,2026-07-04T13:26:38Z,NaN
5,event_markets,cache,0,theoddsapi/sports_soccer_fifa_world_cup_events...,507.881629,200.0,2026-07-04T13:26:38Z,NaN
6,event_odds,cache,0,theoddsapi/sports_soccer_fifa_world_cup_events...,507.881856,200.0,2026-07-04T13:26:38Z,NaN
7,event_odds,cache,0,theoddsapi/sports_soccer_fifa_world_cup_events...,506.882039,200.0,2026-07-04T13:26:38Z,NaN
8,historical_events,dry_run,1,NaN,NaN,NaN,2026-07-04T13:26:38Z,"dry-run; would cost ~1 (1 (metadata snapshot, ..."
9,historical_odds,dry_run,10,NaN,NaN,NaN,2026-07-04T13:26:38Z,dry-run; would cost ~10 (10 × regions × market...


<a id="findings"></a>
## 7 · Findings, caveats, next steps

* Every v4 read endpoint has a coded, guarded path above; the call log
  records exactly what ran, what was served from cache, and what was
  skipped with the reason (budget, dry-run, offline, no key, no event).
* Featured markets come from `/odds` in one cheap call; btts/DNB/props
  are per-event only — cost scales with events × markets, which is why
  the guard exists.
* Historical endpoints cost 10× — they stay in dry-run until
  `RUN_HISTORICAL=True` is set deliberately.
* **Caveat:** quota estimates are documented formulas; observed usage
  (headers) is what the guard trusts for the running total.
* **Next:** notebook 02 builds the Polymarket side; notebook 03 matches
  the two universes.